# NoSQL·Redis 노트북 — 2~6교시

이 노트북은 `NoSQL_Redis_심화.md` **2·3·4교시**의 개념을 실행 가능한 코드로 먼저 정리하고, 이어서 원래 있던 5·6교시(Step 1~4·체크포인트·보너스) 실습으로 연결됩니다. 1교시(복습·도입)는 별도 코드가 필요 없는 텍스트 교시라 여기 포함하지 않습니다. 7교시 개인실습은 `NoSQL_Redis_3_개인실습.ipynb`, Redis 기초(String·Hash·List·TTL) 시연은 `NoSQL_Redis_4_기본_자료구조.ipynb`에 따로 있습니다.

**2·3교시 = 왜 필요한가, 어떤 선택지가 있는가(개념) · 4교시 = 안에서 무슨 일이 일어나는가(원리) · 5·6교시 = 직접 재보고 세 DB를 비교(실습)**


## 2교시 · 왜 관계형 DB 하나로 부족해졌는가

### 모듈 2-1 — 관계형 DB의 한계

관계형 DB는 40년 이상 데이터 저장의 표준이었습니다. 그런데 2000년대 중반부터 인터넷 규모로 성장한 서비스들이 이 표준에 균열을 냈습니다. 균열의 원인은 세 가지입니다 — **규모**, **스키마 변화 속도**, **응답 지연 요구**.

| 문제 | 구체적 상황 | 관계형 DB의 반응 |
|---|---|---|
| **수직 확장의 벽** | 서버 사양을 높이는 데 물리적·비용적 한계 | 가장 좋은 서버 1대로 50만 동접 불가 |
| **수평 확장의 복잡성** | 여러 서버에 나눠야 하는데 ACID 보장이 어려워짐 | 분산 트랜잭션은 성능·복잡도 비용이 큼 |
| **스키마 경직성** | 새 필드 추가 = `ALTER TABLE` = 테이블 잠금 | 1억 행 테이블 컬럼 추가는 수 시간 |
| **JOIN의 비용** | 정규화된 데이터는 읽을 때 JOIN 필요 | 초당 30만 건 조회에서 JOIN은 병목 |

> 아래는 **이 한계를 체감하기 위한 참고용 예시이며 실행하지 않습니다** — `users`·`bookmarks`
> 테이블은 이 실습 환경에 없습니다(4교시에서 MongoDB로 같은 도메인을 다시 설계해봅니다).
> ```sql
> -- 이런 쿼리가 초당 수만 번 실행될 때
> SELECT u.name, d.title, d.content, b.bookmarked_at
> FROM users u
> JOIN bookmarks b ON u.id = b.user_id       -- JOIN 1
> JOIN documents d ON b.doc_id = d.id        -- JOIN 2
> WHERE u.id = $1;
> -- JOIN이 2개, 인덱스가 완벽해도 단일 행 조회에 수십 ms
> -- 수만 건이 동시에 오면?
> ```

> ⚠️ **흔한 실수**: "그러면 PostgreSQL을 쓰면 안 되는 건가요?"가 아닙니다. 이 한계는 **특정 규모와 특정 접근 패턴**에서 나타납니다. 소규모 서비스나 복잡한 트랜잭션이 필요한 시스템에서는 관계형 DB가 여전히 최선입니다.

> 💡 **핵심**: 관계형 DB는 "데이터가 어떤 구조인가"를 중심으로 설계됐습니다. NoSQL은 "데이터를 어떻게 접근하는가"에서 출발합니다. 도서관에 비유하면 관계형 DB는 분류 기준을 먼저 정하고 책을 꽂고, NoSQL은 "이 책을 찾는 가장 빠른 경로"를 먼저 정하고 그에 맞춰 꽂습니다. 단 후자는 "주제별로 책을 모두 가져와"라는 요청에 취약합니다.


In [1]:
# 2교시 참고 — JOIN이 늘어날 때 비용이 어떻게 커지는지 순수 Python으로 체감해봅니다.
# (실제 course_db에 users/bookmarks 테이블은 없으므로, 위 SQL 대신 규모만 시뮬레이션합니다)
import random  # 무작위 값(난수)을 만드는 표준 라이브러리
import time    # 코드 실행에 걸린 시간을 재는 표준 라이브러리

# random.seed(42)로 "난수의 시작점"을 고정하면, 이 코드를 몇 번을 다시 실행해도
# 매번 정확히 같은 "무작위" 값들이 나옵니다 — 42는 특별한 의미 없는 임의의 숫자이고,
# 실습·논문에서 결과를 똑같이 재현하고 싶을 때 관례적으로 이렇게 고정해 둡니다.
random.seed(42)
n_users = 5_000               # 파이썬 숫자 리터럴의 언더스코어(_)는 자릿수 구분용 — 5_000 == 5000
n_bookmarks_per_user = 8      # 사용자 한 명이 갖는 북마크 개수
n_docs = 2_000                # 전체 문서 개수

# 정규화된 구조를 흉내: 사용자 목록, (user_id, doc_id) 북마크 쌍, 문서 목록
# {i: f"user{i}" for i in range(n_users)} 는 "딕셔너리 컴프리헨션" — for문을 한 줄로 압축해
# 딕셔너리를 만드는 파이썬 문법입니다. 풀어 쓰면 아래와 같습니다.
#   users = {}
#   for i in range(n_users):
#       users[i] = f"user{i}"
# 즉 users[0]은 "user0", users[1]은 "user1"... 처럼 "사용자 번호 -> 이름" 매핑을 만듭니다.
users = {i: f"user{i}" for i in range(n_users)}
# [ ... for u in range(n_users) for _ in range(n_bookmarks_per_user) ] 는 "리스트 컴프리헨션"이고,
# for가 두 번 겹친 건 이중 반복문(사용자마다 × 북마크 개수만큼)을 한 줄로 쓴 것입니다.
# 변수명 밑줄(_)은 "이 반복 변수의 값은 실제로 안 쓴다"는 파이썬 관례 표시입니다.
# random.randrange(n_docs)는 0 이상 n_docs 미만의 무작위 정수 하나를 뽑습니다.
# 결과: (사용자번호, 무작위로_고른_문서번호) 쌍들이 담긴 리스트 — "이 사용자가 이 문서를 북마크했다"는 기록입니다.
bookmarks = [
    (u, random.randrange(n_docs))
    for u in range(n_users)
    for _ in range(n_bookmarks_per_user)
]
docs = {i: f"문서{i} 제목" for i in range(n_docs)}  # "문서 번호 -> 제목" 매핑, users와 같은 방식


def join_lookup(target_user_id):
    """JOIN 2개짜리 쿼리를 흉내: 이 사용자의 북마크 -> 각 북마크의 문서 제목까지 조회."""
    # [b for b in bookmarks if b[0] == target_user_id] 는 "조건부 리스트 컴프리헨션" —
    # bookmarks 전체를 하나씩 훑으면서(b), 그 튜플의 첫 번째 값(b[0], 즉 사용자번호)이
    # target_user_id와 같은 것만 골라 새 리스트로 모읍니다. 이 "전체를 한 줄씩 다 훑는" 동작이
    # SQL의 인덱스 없는 JOIN과 똑같은 성격(선형 탐색)이라 JOIN 1에 대응한다고 표시했습니다.
    my_bookmarks = [b for b in bookmarks if b[0] == target_user_id]  # JOIN 1에 해당
    # 위에서 찾은 (사용자번호, 문서번호) 쌍마다, users/docs 딕셔너리에서 실제 이름·제목을 찾아 옵니다.
    # 이 "번호로 딕셔너리를 한 번 더 찾아보는" 단계가 JOIN 2(문서 테이블과의 조인)에 대응합니다.
    return [(users[u], docs[d]) for u, d in my_bookmarks]  # JOIN 2에 해당


# time.perf_counter()는 아주 정밀한 "현재 시각"을 반환합니다 — 시작 시각을 기록해 두고,
# 나중에 다시 호출한 값과 빼면 그 사이에 걸린 시간(초)을 알 수 있습니다.
start = time.perf_counter()
for _ in range(200):  # 200번 조회 = "동시에 여러 요청이 몰릴 때"를 흉내
    join_lookup(random.randrange(n_users))  # 매번 무작위 사용자 한 명을 골라 조회
elapsed = time.perf_counter() - start  # 끝난 시각 - 시작 시각 = 200번 조회에 걸린 전체 시간(초)

# f"...{변수:,}..." 처럼 콜론 뒤에 콤마(,)를 붙이면 숫자에 천 단위 구분 콤마가 자동으로 붙습니다.
# elapsed * 1000 은 초 단위를 밀리초(ms) 단위로 바꾸는 것(1초 = 1000ms), :.1f는 소수점 첫째 자리까지만 표시.
print(f"사용자 {n_users:,}명 × 북마크 {n_bookmarks_per_user}개, 200회 조회: {elapsed * 1000:.1f}ms")
print("-> 인덱스가 없는 선형 탐색이라 사용자 수가 늘수록 조회당 비용이 그대로 늘어납니다.")
print("   실제 PostgreSQL은 인덱스로 이 비용을 크게 줄이지만, JOIN 자체가 사라지진 않습니다.")


사용자 5,000명 × 북마크 8개, 200회 조회: 162.5ms
-> 인덱스가 없는 선형 탐색이라 사용자 수가 늘수록 조회당 비용이 그대로 늘어납니다.
   실제 PostgreSQL은 인덱스로 이 비용을 크게 줄이지만, JOIN 자체가 사라지진 않습니다.


### 모듈 2-2 — NoSQL의 등장: "대체가 아니라 분화"

NoSQL이 등장했을 때 일부는 "관계형 DB의 종말"이라 했습니다. 15년이 지난 지금 그 예언은 틀렸습니다. 대신 데이터 저장 세계는 **분화**했습니다. 금융 거래는 여전히 PostgreSQL에 살고, 실시간 순위는 Redis에, 사용자 프로필은 MongoDB에 삽니다.

**NoSQL이 해결한 세 가지**

```
① 수평 확장 (Scale-Out)      — 서버를 추가할수록 처리량이 선형에 가깝게 증가
② 유연한 스키마               — 문서(또는 행)마다 형태가 달라도 저장 가능
③ 특화된 자료구조             — 데이터 형태에 맞는 엔진 선택 (그래프→그래프DB, 시계열→시계열DB)
```

**무엇을 포기했는가** — NoSQL은 공짜 점심이 아닙니다.

| 포기한 것 | 관계형의 보장 | NoSQL에서 |
|---|---|---|
| 강한 일관성 | 쓰기 직후 어느 서버에서도 같은 값 | 잠깐 다른 값이 보일 수 있음 (최종 일관성) |
| 복잡한 JOIN | 여러 테이블을 자유롭게 조합 | 대부분 단일 컬렉션/키 조회에 특화 |
| 트랜잭션 | ACID 4속성 완전 보장 | 제한적 지원 (MongoDB 4.0+에서 일부) |
| 표준 쿼리 언어 | SQL — 어디서든 같은 문법 | 시스템마다 다른 API |

> 🏭 **실무에서는**: "MongoDB 쓰면 JOIN 못 하잖아요"는 틀린 말이 아닙니다. 하지만 실무에서는 JOIN이 필요한 쿼리를 MongoDB에 날리지 않도록 처음부터 스키마를 그렇게 설계합니다.

> 💡 **핵심**: "NoSQL이냐 관계형이냐"는 틀린 질문입니다. "이 데이터는 어떤 접근 패턴을 가지는가? 스키마가 얼마나 빠르게 바뀌는가? 데이터 규모는 어느 정도인가?" 세 질문에 따라 도구가 결정됩니다. 오늘 배울 PostgreSQL·MongoDB·Redis는 경쟁자가 아니라 하나의 시스템 안에서 역할을 나눠 맡는 팀원들입니다.


## 3교시 · CAP 정리와 데이터 모델 비교

### 모듈 3-1 — CAP 정리: 셋 중 둘만 고를 수 있다

2000년 에릭 브루어가 제시한 CAP 정리는, **일관성(Consistency)·가용성(Availability)·분할 허용성(Partition Tolerance)** 세 가지를 동시에 완전히 보장하는 분산 시스템은 존재하지 않는다는 정리입니다.

| 속성 | 의미 | 실무 번역 |
|---|---|---|
| **C — 일관성** | 모든 노드가 같은 시점에 같은 데이터를 봄 | "어느 서버에 물어봐도 같은 답" |
| **A — 가용성** | 노드 일부가 장애 상태여도 요청에 항상 응답 | "서비스는 멈추지 않음" |
| **P — 분할 허용성** | 네트워크가 분할되어도 시스템이 작동 | "노드 간 통신이 끊겨도 버팀" |

실제 인터넷 환경에서 **P는 선택이 아닙니다.** 네트워크는 반드시 끊기는 순간이 있으므로, 실제 선택지는 **C vs A**입니다.

```
C를 선택하는 경우: 은행 계좌 잔액 조회
  -> 모든 서버가 동의하지 않으면 응답하지 않음
  -> "잠깐 서비스 불가"가 "잘못된 잔액"보다 낫다

A를 선택하는 경우: SNS 피드, 쇼핑 카트
  -> 서버가 끊겨도 "어쩌면 오래된 데이터"로 응답
  -> "잠깐 오래된 피드"가 "서비스 불가"보다 낫다
```

> ⚠️ **흔한 실수**: CAP는 정성적 가이드이지 수학적으로 엄밀한 이분법이 아닙니다. 실제로는 C와 A 사이에 스펙트럼이 존재하며, Cassandra는 `CONSISTENCY LEVEL` 설정으로 요청마다 일관성 수준을 조정할 수 있습니다.

> 💡 **핵심**: CAP 정리는 "어떤 DB가 더 좋은가"가 아니라 **"이 상황에서 무엇을 포기할 수 있는가"를 결정하는 언어**입니다. 잔액이 잠깐 느려도 되지만 틀리면 안 된다면 C를, 좋아요 수가 잠깐 틀려도 되지만 서비스가 멈추면 안 된다면 A를 선택합니다.


In [2]:
# 3교시 참고 — "C를 택하면 응답을 거부, A를 택하면 오래된 값이라도 응답"을 코드로 흉내 냅니다.
# 실제 분산 시스템이 아니라, 두 전략의 '판단 기준' 차이를 아주 작은 시뮬레이션으로 보여줍니다.
import random

random.seed(7)  # 위 셀과 마찬가지로, 매번 같은 "무작위" 결과가 나오도록 시작점을 고정


# 파이썬 함수는 def 함수이름(매개변수들): 로 정의합니다.
# node_is_reachable(불리언 True/False), latest_value(최신 값), stale_value(오래된 값) 세 개를 받아,
# 상황에 맞는 "응답 메시지 문자열"을 return(반환)합니다 — 아직 화면에 출력하는 게 아니라 값을 돌려주는 것뿐입니다.
def read_with_cp(node_is_reachable, latest_value, stale_value):
    """CP 전략(예: 은행 잔액) — 최신 노드에 닿지 못하면 응답 자체를 거부합니다."""
    if not node_is_reachable:  # not True는 False, not False는 True — 즉 "연결이 안 됐다면"
        return "[거부] 다른 노드와 합의를 확인할 수 없어 응답하지 않음"
    # 위 if에서 return으로 이미 함수가 끝났으므로, 여기 도달했다는 건 연결이 됐다는 뜻입니다.
    return f"[응답] {latest_value}"


def read_with_ap(node_is_reachable, latest_value, stale_value):
    """AP 전략(예: SNS 피드) — 최신 노드에 못 닿아도 갖고 있는 값으로 일단 응답합니다."""
    if not node_is_reachable:
        # CP와 다른 점: 거부하지 않고, 갖고 있던(오래됐을 수 있는) 값으로라도 응답합니다.
        return f"[응답, 오래됐을 수 있음] {stale_value}"
    return f"[응답] {latest_value}"


print("네트워크 분할 상황을 10번 시뮬레이션 (노드 연결 여부를 무작위로 흔들어봅니다)")
# f"{문자열:<10}" 은 문자열을 왼쪽 정렬하고 10칸 너비로 맞추는 서식 지정입니다(오른쪽에 공백 채움) —
# 표처럼 줄이 맞게 출력하려고 씁니다.
print(f"{'상황':<10}{'CP(계좌 잔액)':<45}{'AP(SNS 피드)'}")
for i in range(10):  # 0부터 9까지 총 10번 반복
    # random.random()은 0.0 이상 1.0 미만의 무작위 실수를 뽑습니다.
    # 그 값이 0.4보다 크면(60% 확률) True(연결됨), 0.4 이하이면(40% 확률) False(분할 발생)로 취급합니다.
    reachable = random.random() > 0.4  # 40% 확률로 네트워크 분할 발생
    label = "정상" if reachable else "분할 발생"  # 파이썬의 "조건부 표현식": 조건이 참이면 앞 값, 거짓이면 뒤 값
    cp = read_with_cp(reachable, latest_value="잔액 120,000원", stale_value="잔액 120,000원(5분 전)")
    ap = read_with_ap(reachable, latest_value="좋아요 342개", stale_value="좋아요 338개(5분 전)")
    print(f"{label:<10}{cp:<45}{ap}")

print("\n-> CP는 분할 발생 시 '거부'만 하고, AP는 분할 발생 시에도 (오래된) 값으로 계속 응답합니다.")


네트워크 분할 상황을 10번 시뮬레이션 (노드 연결 여부를 무작위로 흔들어봅니다)
상황        CP(계좌 잔액)                                    AP(SNS 피드)
분할 발생     [거부] 다른 노드와 합의를 확인할 수 없어 응답하지 않음             [응답, 오래됐을 수 있음] 좋아요 338개(5분 전)
분할 발생     [거부] 다른 노드와 합의를 확인할 수 없어 응답하지 않음             [응답, 오래됐을 수 있음] 좋아요 338개(5분 전)
정상        [응답] 잔액 120,000원                             [응답] 좋아요 342개
분할 발생     [거부] 다른 노드와 합의를 확인할 수 없어 응답하지 않음             [응답, 오래됐을 수 있음] 좋아요 338개(5분 전)
정상        [응답] 잔액 120,000원                             [응답] 좋아요 342개
분할 발생     [거부] 다른 노드와 합의를 확인할 수 없어 응답하지 않음             [응답, 오래됐을 수 있음] 좋아요 338개(5분 전)
분할 발생     [거부] 다른 노드와 합의를 확인할 수 없어 응답하지 않음             [응답, 오래됐을 수 있음] 좋아요 338개(5분 전)
정상        [응답] 잔액 120,000원                             [응답] 좋아요 342개
분할 발생     [거부] 다른 노드와 합의를 확인할 수 없어 응답하지 않음             [응답, 오래됐을 수 있음] 좋아요 338개(5분 전)
정상        [응답] 잔액 120,000원                             [응답] 좋아요 342개

-> CP는 분할 발생 시 '거부'만 하고, AP는 분할 발생 시에도 (오래된) 값으로 계속 응답합니다.


### 모듈 3-2 — 데이터 모델 비교

NoSQL은 단일 기술이 아니라 목적에 따른 여러 모델의 집합입니다. 오늘 배울 MongoDB는 문서형, Redis는 Key-Value 모델입니다.

**데이터 모델 지형도**

| 모델 | 대표 시스템 | 저장 단위 | 최적 패턴 | 취약 패턴 |
|---|---|---|---|---|
| **관계형** | PostgreSQL, MySQL | 행(Row) | 복잡한 JOIN·트랜잭션 | 수평 확장·스키마 잦은 변화 |
| **문서형** | MongoDB, CouchDB | 문서(JSON/BSON) | 계층 구조·다양한 스키마 | 여러 컬렉션 JOIN |
| **Key-Value** | Redis, DynamoDB | 키에 값 | 단순 조회·캐시·세션 | 부분 검색·범위 쿼리 |
| **컬럼형** | Cassandra, HBase | 열 군(Column Family) | 시계열·대용량 쓰기 | 작은 데이터·복잡한 집계 |
| *(11월에 채웁니다)* | — | — | — | — |

> 🏭 **실무에서는**: "MongoDB는 스키마가 없다"는 말을 "설계가 필요 없다"로 오해하는 경우가 많습니다. 스키마를 DB가 강제하지 않을 뿐, 애플리케이션 레이어에서의 일관된 구조 설계는 필수입니다.

> 💡 **핵심**: 데이터 모델 선택은 "어떤 질문을 자주 던지는가"에 달려 있습니다. "사용자 ID 하나로 프로필 전체를 빠르게 가져오고 싶다"면 문서형, "실시간 좋아요 순위 상위 10개"면 Key-Value + Sorted Set, "이 거래가 완전히 성공하거나 완전히 실패해야 한다"면 관계형이 자연스럽습니다.

이어지는 4교시에서 문서형(MongoDB)과 Key-Value(Redis)를 직접 코드로 만나봅니다.


## 4교시 · MongoDB와 Redis의 안쪽

### 모듈 4-1 — MongoDB: 임베딩 vs 참조

MongoDB 모델링에서 가장 중요한 결정은 **임베딩(Embedding)** 과 **참조(Reference)** 중 어느 것을 선택하는가입니다. 주의: 여기서 "임베딩"은 벡터 임베딩이 아니라, 다른 문서의 데이터를 현재 문서 **안에** 직접 집어넣는 것을 의미합니다.

아래에서 사용자가 문서를 북마크하는 같은 도메인을 두 가지 방식으로 실제로 만들어 비교합니다 (2교시에서 SQL로 봤던 `users`·`bookmarks`·`documents` 관계를 MongoDB로 옮긴 것입니다).


In [3]:
# 4교시부터 바로 실행해도 되도록, 이 노트북에서 나중에(Step 1 이후) 쓰는 run_redis()/mongo_db를
# 여기서 먼저 준비합니다. 아래 '환경 준비' 셀(Step 1 앞)에서 다시 한번 정의하지만 내용은 동일하니,
# 그 셀을 이미 실행했어도 이 셀을 실행해도 안전합니다(재정의만 될 뿐 값은 그대로입니다).
import subprocess  # 터미널 명령(docker, redis-cli 등)을 파이썬에서 대신 실행해주는 표준 라이브러리
import mongomock   # "진짜 MongoDB" 대신 파이썬 메모리 안에서 흉내 내주는 외부 라이브러리

# 💡 4교시부터 바로 시작해도 되도록, db-redis가 아직 없으면 여기서 새로 만들고 시작까지 보장합니다
#    (이미 떠 있으면 곧바로 넘어갑니다 — 재실행 안전).
if subprocess.run(["docker", "inspect", "db-redis"], capture_output=True, text=True).returncode != 0:
    subprocess.run(["docker", "run", "-d", "--name", "db-redis", "-p", "6379:6379", "redis:8"],
                    capture_output=True, text=True)
subprocess.run(["docker", "start", "db-redis"], capture_output=True, text=True)


# *args 는 "개수가 정해지지 않은 여러 개의 인자를 하나로 묶어서 받는다"는 뜻입니다.
# 예를 들어 run_redis("get", "user:u001") 라고 호출하면 args는 ("get", "user:u001") 튜플이 됩니다.
def run_redis(*args):
    """redis-cli subprocess 헬퍼 — db-redis 컨테이너 안의 redis-cli로 실제 Redis에 실행합니다."""
    # ["docker", "exec", "db-redis", "redis-cli"] + [str(a) for a in args] 는 두 리스트를 이어붙이는 것(+)입니다.
    # [str(a) for a in args]는 args의 각 값을 문자열로 바꾼 새 리스트를 만드는 리스트 컴프리헨션입니다
    # (redis-cli 인자는 모두 문자열이어야 하므로, 숫자를 넣어도 안전하게 문자열로 변환합니다).
    # 💡 redis-cli가 이 컴퓨터가 아니라 db-redis 컨테이너 "안"에 있으므로, docker exec로 그 안에 들어가
    #    실행합니다 — 이 컴퓨터에 redis-cli를 따로 설치할 필요가 없습니다.
    # 예: run_redis("set", "key", 123) -> cmd = ["docker", "exec", "db-redis", "redis-cli", "set", "key", "123"]
    cmd = ["docker", "exec", "db-redis", "redis-cli"] + [str(a) for a in args]
    result = subprocess.run(cmd, capture_output=True, text=True)
    out = result.stdout.strip()  # 앞뒤 공백·줄바꿈 제거
    # Redis는 "값이 없음"을 "(nil)"이라는 특수 문자열로 표시합니다.
    # 이걸 그대로 두면 파이썬에서 "값이 있다"고 착각하기 쉬우므로, 여기서 진짜 파이썬의 None으로 바꿔줍니다.
    return None if out == "(nil)" else out


# mongomock.MongoClient()는 "MongoDB 서버에 접속하는 창구"를 만드는 것입니다(실제로는 메모리 안에서 흉내).
mongo_client = mongomock.MongoClient()
mongo_db = mongo_client["course_db"]  # 이 창구 안에서 "course_db"라는 이름의 데이터베이스를 선택
print("Redis 연결 확인:", run_redis("ping") == "PONG")  # ping 응답이 정확히 "PONG"인지 비교 -> True/False
print("mongomock 준비 완료:", mongo_db.name)


Redis 연결 확인: True
mongomock 준비 완료: course_db


In [4]:
# ✅ 방법 1: 임베딩(Embedding) — 데이터를 문서 안에 직접 내장
# mongo_db.users_v1 은 "course_db 안의 users_v1이라는 컬렉션"을 가리킵니다 — SQL의 테이블과 비슷한 개념입니다.
# .delete_many({}) 는 조건 없이({} = 빈 조건 = 전체) 모든 문서를 지웁니다.
# 이 노트북 셀을 여러 번 실행해도 매번 깨끗한 상태에서 시작하도록 미리 비워 두는 것입니다.
mongo_db.users_v1.delete_many({})  # 재실행 대비 초기화

# .insert_one({...}) 은 중괄호 안의 딕셔너리 하나를 문서(document) 하나로 저장합니다.
# MongoDB에서 "_id"는 그 문서의 고유 식별자 필드입니다(SQL의 PRIMARY KEY와 같은 역할) —
# 직접 값을 지정하지 않으면 MongoDB가 자동으로 만들어 주지만, 여기서는 "user001"로 직접 지정했습니다.
mongo_db.users_v1.insert_one({
    "_id": "user001",
    "name": "김철수",
    "email": "kim@example.com",
    # "bookmarks" 값이 [ {...}, {...} ] 처럼 딕셔너리들의 리스트라는 점이 이 셀의 핵심입니다 —
    # 사용자 문서 "안에" 북마크 정보(딕셔너리)를 통째로 넣은 것이 바로 "임베딩"입니다.
    "bookmarks": [                        # <- 북마크를 사용자 문서 안에 직접 내장
        {
            "doc_id": "A001",
            "title": "2026 반도체 시장 전망",   # 제목도 복사해서 넣음
            "bookmarked_at": "2026-09-22T10:30:00Z",
        },
        {
            "doc_id": "B002",
            "title": "AI 규제 동향",
            "bookmarked_at": "2026-09-22T14:20:00Z",
        },
    ],
})

# .find_one({조건}) 은 조건에 맞는 문서를 딱 하나만 찾아 돌려줍니다(SQL의 SELECT ... LIMIT 1과 비슷).
# {"_id": "user001"} 조건은 "_id 필드가 정확히 user001인 문서"를 뜻합니다.
# 임베딩 방식이므로 이 한 번의 호출만으로 사용자 정보 + 북마크 목록이 전부 함께 돌아옵니다.
result = mongo_db.users_v1.find_one({"_id": "user001"})
print("조회 횟수: 1회 (findOne 한 번)")
print("사용자:", result["name"])
# for b in result["bookmarks"]: 는 result 딕셔너리 안의 "bookmarks" 리스트를 하나씩 꺼내 b에 담아 반복합니다.
for b in result["bookmarks"]:
    print("  -", b["title"], "(", b["bookmarked_at"], ")")


조회 횟수: 1회 (findOne 한 번)
사용자: 김철수
  - 2026 반도체 시장 전망 ( 2026-09-22T10:30:00Z )
  - AI 규제 동향 ( 2026-09-22T14:20:00Z )


In [5]:
# ✅ 방법 2: 참조(Reference) — ID만 저장하고 별도 조회
mongo_db.users_v2.delete_many({})
mongo_db.documents_v2.delete_many({})

# 사용자 컬렉션 — ID만 저장
# 위 임베딩 방식과 다르게, bookmarks 안에 제목 등 상세 정보를 복사해 넣지 않고
# "bookmark_ids"라는 이름으로 문서 ID(문자열)만 리스트로 저장합니다 — 진짜 정보는 다른 컬렉션에 있습니다.
mongo_db.users_v2.insert_one({
    "_id": "user001",
    "name": "김철수",
    "email": "kim@example.com",
    "bookmark_ids": ["A001", "B002"],
})

# 문서(documents) 컬렉션 — 별도
# .insert_many([...]) 는 .insert_one과 달리 여러 문서를 리스트로 한 번에 저장합니다.
mongo_db.documents_v2.insert_many([
    {"_id": "A001", "title": "2026 반도체 시장 전망", "created_at": "2026-09-10"},
    {"_id": "B002", "title": "AI 규제 동향", "created_at": "2026-09-12"},
])

# 조회: 사용자 먼저 -> bookmark_ids 추출 -> 각 문서 별도 조회 (쿼리 N+1)
# query_count는 "DB를 실제로 몇 번 조회했는지" 손으로 세어보기 위한 변수입니다.
query_count = 0

user = mongo_db.users_v2.find_one({"_id": "user001"})  # 1번째 조회: 사용자 문서
query_count += 1  # += 1 은 query_count = query_count + 1 과 같습니다(기존 값에 1을 더해 다시 저장)

titles = []
# user["bookmark_ids"]는 ["A001", "B002"] 같은 리스트입니다. for로 하나씩(doc_id) 꺼내면서
# 그 ID로 documents_v2에서 문서를 또 조회합니다 — 이것이 "북마크 개수만큼 추가로 조회"하는 부분입니다.
for doc_id in user["bookmark_ids"]:
    doc = mongo_db.documents_v2.find_one({"_id": doc_id})  # 2번째, 3번째... 조회: 문서마다 하나씩
    query_count += 1
    titles.append(doc["title"])  # .append()는 리스트 끝에 값 하나를 추가하는 메서드

# "N+1"이라는 이름의 유래: 사용자 조회 1번(N과 무관한 고정 1회) + 북마크 개수(N)만큼의 추가 조회 = 총 N+1회.
# f-string 안의 {len(user['bookmark_ids'])}는 그 리스트의 길이(개수)를 구하는 len() 함수를 바로 넣은 것입니다.
print(f"조회 횟수: {query_count}회 (사용자 1회 + 문서 {len(user['bookmark_ids'])}회 = N+1)")
print("사용자:", user["name"])
for t in titles:
    print("  -", t)


조회 횟수: 3회 (사용자 1회 + 문서 2회 = N+1)
사용자: 김철수
  - 2026 반도체 시장 전망
  - AI 규제 동향


**판단 기준 3가지**

| 질문 | 임베딩 선택 | 참조 선택 |
|---|---|---|
| ① 함께 읽히는가? | 사용자 조회 시 항상 북마크도 함께 필요 | 북마크를 따로 조회하는 경우가 많음 |
| ② 무한히 늘어나는가? | 북마크가 최대 수십 개로 제한적 | 댓글·이력이 수만 건으로 증가 가능 |
| ③ 독립적으로 수정되는가? | 북마크 안의 제목을 독립 수정할 일 없음 | 원본 문서 제목이 바뀌면 모든 북마크도 갱신 필요 |

> ⚠️ **흔한 실수**: "MongoDB는 유연하니까 일단 다 집어넣자"가 임베딩 과다 사용의 원인입니다. 한 문서에 수백 개의 중첩 문서가 들어가면 조회는 빠르지만 업데이트와 재설계가 매우 어려워집니다. (단일 문서 크기 제한도 16MB로 정해져 있습니다.)

> 💡 **핵심**: MongoDB의 임베딩 vs 참조는 서류를 한 봉투에 다 넣을지, 링크만 걸어둘지 같은 결정입니다. 항상 함께 꺼낼 서류라면 한 봉투에 넣는 게 빠르지만, 그 묶음이 수천 장으로 늘어나면 봉투가 찢어집니다. 여러 봉투에 같은 서류 사본이 들어 있으면 원본이 바뀔 때마다 모든 봉투를 열어 사본을 고쳐야 합니다.

---

### 모듈 4-2 — Redis: 자료구조별 용도

Redis를 "빠른 캐시"로만 아는 경우가 많습니다. 하지만 Redis는 **5가지 핵심 자료구조**를 내장한 인메모리 데이터베이스입니다. 각 자료구조는 특정 접근 패턴에서 O(1) 또는 O(log n)을 보장하도록 설계됐습니다.

> String→캐시, Hash→세션, List→큐 로 정확히 대응됩니다. 아래에서 5종을 실제 `redis-cli`로 하나씩 실행해봅니다 (String·Hash·List를 `redis-py`로 손으로 더 천천히 익히고 싶다면 `NoSQL_Redis_4_기본_자료구조.ipynb`를 참고하세요).


In [ ]:
# 4교시 참고 — Redis 5대 자료구조를 바로 위 셀에서 준비한 run_redis() 헬퍼로 하나씩 실행해봅니다.
# run_redis(*args)는 "redis-cli" 뒤에 args를 그대로 이어붙여 실행하는 함수였습니다.
# 즉 run_redis("set", "key", "value")는 터미널에서 `redis-cli set key value`를 친 것과 같습니다.
# 아래 각 블록은 Redis 명령 하나(SET/HSET/LPUSH/SADD/ZADD 등)를 자료구조별로 시연합니다.


In [6]:
# 1. String — 가장 단순, 범용 캐시·세션
# `SET key value EX 1800` 형태: 세 번째 인자(값) 뒤에 "ex"(EXpire의 줄임말)와 "1800"(초 단위)을 붙이면
# 1800초(=30분) 뒤 Redis가 이 키를 자동으로 지워줍니다 — 로그인 세션처럼 "일정 시간 뒤 만료"가 필요할 때 씁니다.
run_redis("set", "user:u001:session", "eyJhbGci...", "ex", "1800")
print("String:", run_redis("get", "user:u001:session"))  # get으로 방금 저장한 값을 그대로 조회


String: eyJhbGci...


In [10]:
run_redis("set", "user:u002:session", "eyJhbGci...", "ex", "1")


'OK'

In [ ]:
#1초만 유지되었기 때문에 u002는 아무것도 안 나옴
print("String:", run_redis("get", "user:u002:session"))

String: 


In [12]:
# 2. Hash — 객체를 필드별로 저장
# Hash는 파이썬의 딕셔너리 하나를 통째로 저장하는 것과 비슷합니다 — 하나의 키(user:u001:profile) 안에
# name/tier/last_login 같은 "필드 이름 -> 값" 쌍을 여러 개 담습니다.
run_redis("del", "user:u001:profile")  # 이전 실습 흔적이 남아있을 수 있으므로 먼저 삭제(del)하고 새로 만듭니다
# hset key 필드1 값1 필드2 값2 필드3 값3 ... 처럼 필드-값을 계속 나열해서 한 번에 여러 필드를 설정합니다
run_redis("hset", "user:u001:profile", "name", "김철수", "tier", "premium", "last_login", "2026-09-23")
print("Hash (tier만):", run_redis("hget", "user:u001:profile", "tier"))  # hget으로 필드 하나만 콕 집어 조회

Hash (tier만): premium


In [ ]:
#괄호 안에 다양한 것들을 넣어볼 수 있음
print("Hash (last_login만):", run_redis("hget", "user:u001:profile", "last_login"))

Hash (last_login만): 2026-09-23


In [16]:
# 3. List — 최근 N개 기록, 큐
# List는 파이썬의 리스트처럼 "순서가 있는 값들의 나열"입니다. lpush는 Left-PUSH, 즉 리스트의 왼쪽(맨 앞)에
# 새 값을 끼워 넣습니다 — 그래서 나중에 lpush한 값일수록 맨 앞(최신)에 위치하게 됩니다.
run_redis("del", "user:u001:recent_demo")
run_redis("lpush", "user:u001:recent_demo", "A001")  # 왼쪽에 추가 (최신이 앞)
run_redis("lpush", "user:u001:recent_demo", "B002")  # 이번엔 B002가 A001보다 더 왼쪽(더 앞)으로 들어감
# lrange key 시작인덱스 끝인덱스: 0번째부터 4번째까지 꺼내옵니다(리스트 길이가 짧으면 있는 만큼만 반환).
# run_redis()가 리스트 전체를 하나의 문자열로 돌려주므로, .splitlines()로 줄 단위로 잘라 파이썬 리스트로 만듭니다.
print("List (최근 순):", run_redis("lrange", "user:u001:recent_demo", "0", "4").splitlines())

List (최근 순): ['B002', 'A001']


In [17]:
# 4. Set — 중복 없는 집합, 멤버십 O(1) 확인
# Set은 파이썬의 set()과 같은 개념 — 순서 없이, 중복 없이 값을 모아두고 "이 값이 들어있나?"를 매우 빠르게 확인합니다.
run_redis("del", "doc:A001:viewers_demo")
run_redis("sadd", "doc:A001:viewers_demo", "u001", "u002", "u003")  # sadd: Set에 값 여러 개를 한 번에 추가
# sismember key value: value가 이 Set 안에 있으면 "1"(True에 해당), 없으면 "0"(False에 해당)을 문자열로 반환
print("Set 멤버십(u004):", run_redis("sismember", "doc:A001:viewers_demo", "u004"), "(0=False)")

Set 멤버십(u004): 0 (0=False)


In [ ]:
#존재하는 걸 넣으면 1(True) 출력
print("Set 멤버십(u001):", run_redis("sismember", "doc:A001:viewers_demo", "u001"), "(0=False)")

Set 멤버십(u001): 1 (0=False)


In [19]:
# 5. Sorted Set — 점수가 있는 집합, 순위 조회
# Sorted Set은 Set과 비슷하지만 각 값마다 "점수(score)"가 붙어서, 점수 순으로 자동 정렬된 상태를 유지합니다
# — 리더보드(순위표)나 "점수 높은 순 Top N 조회" 같은 기능에 적합합니다.
run_redis("del", "doc:scores_demo")
# zadd key 점수1 값1 점수2 값2 ...: A001은 점수 95.5, B002는 87.2, C003은 91.0으로 각각 등록
run_redis("zadd", "doc:scores_demo", "95.5", "A001", "87.2", "B002", "91.0", "C003")
# zrevrange key 시작 끝 withscores: REV는 REVerse(역순), 즉 점수가 높은 것부터 내림차순으로 꺼냅니다.
# withscores를 붙이면 값과 점수가 번갈아(A001, 95.5, C003, 91.0, ...) 함께 반환됩니다.
print("Sorted Set (점수 내림차순):", run_redis("zrevrange", "doc:scores_demo", "0", "9", "withscores").splitlines())

Sorted Set (점수 내림차순): ['A001', '95.5', 'C003', '91', 'B002', '87.2']


**자료구조 선택 가이드**

| 용도 | 추천 | 이유 |
|---|---|---|
| API 응답 캐시, 세션 토큰 | String | 단순 get/set, O(1) |
| 사용자 프로필 (여러 필드) | Hash | 필드별 개별 접근·수정 가능 |
| 최근 조회 기록, 작업 큐 | List | 순서 보장, FIFO/LIFO |
| 팔로워 집합, 방문자 확인 | Set | 중복 제거, 집합 연산(교집합 등) |
| 실시간 순위, 레이팅 | Sorted Set | 점수 기반 정렬, O(log n) 삽입 |

> ⚠️ **흔한 실수**: `setex(key, 초, value)`는 deprecated입니다 — 항상 `set(key, value, ex=초)` 형태를 쓰세요. Sorted Set을 모른 채 "실시간 좋아요 순위 상위 10개"를 String에 JSON으로 저장하고 Python에서 매번 정렬하는 구현도 흔한 실수입니다 — `ZRANGE` 한 번으로 해결됩니다.

> 💡 **핵심**: Redis 자료구조 선택 기준은 "어떤 연산이 O(1)이어야 하는가"입니다. 특정 사용자가 특정 문서를 봤는지 확인(O(1))이면 Set, 상위 10위 문서(O(log n))면 Sorted Set, 사용자 정보를 특정 필드만 수정(O(1))이면 Hash입니다. 단 Redis는 인메모리이므로 재시작 시 데이터가 사라질 수 있습니다 — 영속성 설정(RDB·AOF)을 확인해야 합니다.

**4교시 마무리 — 5교시에서 확인할 가설**

- **가설 1**: MongoDB에서 임베딩으로 설계된 북마크 조회는 참조 설계보다 DB 접근 횟수가 적다 (방금 위에서 1회 vs N+1회로 직접 확인했습니다).
- **가설 2**: Redis Sorted Set으로 문서 순위를 조회하면 PostgreSQL의 `ORDER BY + LIMIT`보다 응답이 빠르다.
- **가설 3**: Cache-Aside(지연 로딩) 전략에서 첫 번째 조회는 반드시 DB를 거치고, 이후 캐시 히트 시에는 DB에 접근하지 않는다.

이 세 가설을 아래 Step 1부터 직접 실측하며 검증합니다.

---


# NoSQL·Redis 핵심개념 데모

> **오늘은 PostgreSQL·Redis는 100% 실제 서버로, MongoDB는 mongomock(pymongo와 거의 동일한 API로 동작하는 실제 파이썬 라이브러리)으로 진행합니다.**
> mongomock은 실제 서버가 아니라 인메모리 시뮬레이션이지만, 지난 버전의 손수 제작 `FakeMongoCollection`과 달리 `insert_one().inserted_id`, 배열 필드 검색(`find({"tags": "값"})`) 등 pymongo의 실제 동작을 정확히 재현합니다.

## 실행 안내
- 🐳 **선행 조건**: `db-pg`(PostgreSQL, course_db)와 `db-redis`가 떠 있어야 합니다 — `NoSQL_Redis_1_환경확인.ipynb`에서 이미 확인했다면 그대로 진행하세요.
- `run_sql()`/`run_redis()`는 지난 실습과 같은 subprocess 헬퍼입니다. Redis 기초(String·Hash·List·TTL)를 real `redis-py` 클라이언트로 먼저 익히고 싶다면 `NoSQL_Redis_4_기본_자료구조.ipynb`를 참고하세요 — 이 노트북은 지난 실습과의 일관성을 위해 subprocess 방식을 계속 씁니다.
- 위에서 아래로 순서대로 실행하세요. 각 Step은 재실행해도 안전하도록 초기화 코드를 포함합니다.
- 결과값은 PostgreSQL 16 · Redis 7.0.15 · mongomock 4.3.0 기준으로 실측한 **확정값**입니다.
- 이 노트북은 **2교시 개념 정리부터 6교시(Step 1~4·체크포인트·보너스)까지**를 다룹니다. 7교시 개인실습은 `NoSQL_Redis_3_개인실습.ipynb`에서 이어집니다.


In [30]:
# ✅ 포인트: 지난 실습과 같은 run_sql()/run_redis() subprocess 헬퍼 + 오늘 새로 쓰는 mongomock
# (이 셀 하나로 노트북이 완결됩니다 — 별도 파일이 필요 없습니다)
# subprocess: 터미널 명령(docker, psql, redis-cli)을 파이썬 안에서 대신 실행해주는 표준 라이브러리
# json/time: 이 셀 자체에는 안 쓰이지만 아래 다른 셀들이 공용으로 쓰므로 여기서 한 번에 import
# io: 문자열(텍스트)을 "파일처럼" 다루게 해주는 표준 라이브러리 — 아래 pd.read_csv(io.StringIO(...))에서 사용
# os: 환경변수(비밀번호 등)를 읽고 설정하는 표준 라이브러리
import subprocess, json, time, io, os
import pandas as pd   # 표 형태(행·열) 데이터를 다루는 대표적인 외부 라이브러리 — SQL 결과를 표로 받을 때 사용
import mongomock       # "진짜 MongoDB" 대신 파이썬 메모리 안에서 흉내 내주는 외부 라이브러리

# 💡 이 노트북을 NoSQL_Redis_1_환경확인.ipynb 없이 단독으로 열어도 동작하도록, db-pg 시작 + db-redis
#    생성(없을 때만)·시작을 여기서 한 번 더 보장합니다. 이미 떠 있으면 순식간에 끝납니다(재실행 안전).
if subprocess.run(["docker", "inspect", "db-redis"], capture_output=True, text=True).returncode != 0:
    subprocess.run(["docker", "run", "-d", "--name", "db-redis", "-p", "6379:6379", "redis:8"],
                    capture_output=True, text=True)
subprocess.run(["docker", "start", "db-pg", "db-redis"], capture_output=True, text=True)

# 함수 안에 """ ... """로 감싼 첫 문장은 "docstring"이라고 부르며, 이 함수가 무슨 일을 하는지 설명하는
# 공식 주석입니다 — help(run_sql)을 실행하면 이 내용이 그대로 출력됩니다.
def run_sql(sql, as_df=True):
    """psql subprocess 헬퍼 — db-pg 컨테이너 안의 psql로 실제 PostgreSQL(course_db)에 실행합니다."""
    # as_df=True 처럼 매개변수에 기본값을 미리 지정해두면, 호출할 때 그 인자를 생략해도 됩니다
    # (예: run_sql("SELECT 1;") 라고만 써도 as_df=True가 자동 적용됩니다).
    # 💡 psql이 이 컴퓨터(호스트)가 아니라 db-pg 컨테이너 "안"에 있으므로, docker exec로 그 안에 들어가
    #    실행합니다 — 이 컴퓨터에 psql을 따로 설치할 필요가 없습니다. -e PGPASSWORD=...는 컨테이너 실행에
    #    비밀번호를 넣어주는 옵션입니다(psql이 비밀번호를 묻지 않고 자동으로 읽어갑니다).
    if as_df:
        # -A(정렬 없이) -F ","(구분자를 콤마로) 옵션으로 CSV 형태 텍스트를 뽑아내는 psql 명령 조합입니다.
        cmd = ["docker", "exec", "-e", "PGPASSWORD=postgres", "db-pg", "psql", "-U", "postgres", "-d", "course_db",
               "-v", "ON_ERROR_STOP=1", "-P", "footer=off", "-A", "-F", ",", "-c", sql]
        result = subprocess.run(cmd, capture_output=True, encoding="utf-8", errors="replace")
        if result.returncode != 0:  # 0이 아니면 psql 실행이 실패했다는 뜻 — 에러 메시지를 그대로 올려서 알림
            raise RuntimeError(result.stderr)
        text = result.stdout.strip("\n")  # 맨 앞뒤의 줄바꿈만 제거(가운데 내용은 그대로 둠)
        # io.StringIO(text)는 문자열 text를 "디스크에 저장된 CSV 파일"인 것처럼 감싸줍니다.
        # pd.read_csv()는 원래 파일 경로를 받는 함수인데, 이렇게 하면 파일을 실제로 만들지 않고도
        # SQL 결과 텍스트를 곧바로 표(DataFrame)로 변환할 수 있습니다 — text가 비어있으면 빈 표를 반환.
        return pd.read_csv(io.StringIO(text)) if text else pd.DataFrame()
    else:
        cmd = ["docker", "exec", "-e", "PGPASSWORD=postgres", "db-pg", "psql", "-U", "postgres", "-d", "course_db",
               "-v", "ON_ERROR_STOP=1", "-c", sql]
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0:
            raise RuntimeError(result.stderr)
        return result.stdout.strip()

# *args 는 "개수가 정해지지 않은 여러 개의 인자를 하나로 묶어서 받는다"는 뜻입니다.
def run_redis(*args):
    """redis-cli subprocess 헬퍼 — db-redis 컨테이너 안의 redis-cli로 실제 Redis에 실행합니다."""
    # [str(a) for a in args]는 args의 각 값을 문자열로 바꾼 새 리스트를 만드는 리스트 컴프리헨션입니다
    # (redis-cli 인자는 모두 문자열이어야 하므로, 숫자를 넣어도 안전하게 문자열로 변환합니다).
    # 💡 redis-cli도 이 컴퓨터가 아니라 db-redis 컨테이너 안에 있으므로 docker exec로 실행합니다.
    cmd = ["docker", "exec", "db-redis", "redis-cli"] + [str(a) for a in args]
    result = subprocess.run(cmd, capture_output=True, encoding="utf-8", errors="replace")
    out = result.stdout.strip()
    # Redis는 "값이 없음"을 "(nil)"이라는 특수 문자열로 표시합니다 — 이걸 파이썬의 진짜 None으로 바꿔줍니다.
    return None if out == "(nil)" else out

# mongomock.MongoClient()는 "MongoDB 서버에 접속하는 창구"를 만드는 것입니다(실제로는 메모리 안에서 흉내).
mongo_client = mongomock.MongoClient()
mongo_db = mongo_client["course_db"]  # 이 창구 안에서 "course_db"라는 이름의 데이터베이스를 선택

# run_sql("SELECT 1;", as_df=False)는 접속만 확인하는 용도라 결과 텍스트 내용 자체는 중요하지 않고,
# "에러 없이 빈 문자열이 아닌 뭔가가 돌아왔는가"만 봅니다. == ""는 "결과가 빈 문자열과 같은가"를 판정하고,
# 맨 앞의 not이 그 판정을 뒤집습니다 — 즉 "결과가 비어있지 않다"(=연결이 정상이다)를 최종적으로 확인합니다.
print("PostgreSQL 연결 확인:", not run_sql("SELECT 1;", as_df=False) == "")
print("Redis 연결 확인:", run_redis("ping") == "PONG")  # ping 응답이 정확히 "PONG"인지 비교 -> True/False
print("mongomock 준비 완료:", mongo_db.name)


PostgreSQL 연결 확인: True
Redis 연결 확인: True
mongomock 준비 완료: course_db


## Step 1 · Redis 자료구조 — 오늘 나중에 재사용할 키 만들기

6교시에서 재사용할 List(`recent:u1001`)와, 4교시 이론에서 다룬 Sorted Set(`doc:scores`) 예시를 실제로 만들어봅니다.

> 🔬 **참고 — redis-cli 원문 응답의 실제 모양**: `run_redis()`는 `redis-cli`가 돌려주는 텍스트를 그대로 반환합니다. List·Sorted Set처럼 값이 여러 개인 응답은 **줄바꿈으로 이어진 하나의 문자열**로 오므로(Python list가 아님) `.splitlines()`로 직접 나눠야 합니다. 점수(score)도 문자열로 오는데, `87.2`가 `87.200000000000003`처럼 부동소수점 오차가 그대로 보이는 경우가 있습니다 — 실수가 아니라 IEEE754 배정밀도 부동소수점의 실제 동작입니다. `redis-py`(`NoSQL_Redis_4_기본_자료구조.ipynb`)처럼 제대로 된 클라이언트를 쓰면 이런 원문 처리를 클라이언트가 대신 해줍니다.


In [31]:
# ✅ Step 1-A: List — 사용자 1001의 최근 조회 상품 (6교시에서 재사용)
run_redis("del", "recent:u1001")
# rpush는 Right-PUSH, 즉 리스트의 오른쪽(맨 뒤)에 값을 추가합니다 — 앞서 4교시의 lpush(왼쪽 추가)와 반대 방향입니다.
# 여기서는 "조회한 순서 그대로" 뒤에 쌓아서, 최근 본 상품 목록을 시간순으로 기록합니다.
run_redis("rpush", "recent:u1001", "product:501", "product:502", "product:503")
# lrange key 0 -1: "-1"은 Redis에서 "마지막 인덱스(끝까지)"를 뜻하는 특수 값입니다.
# 즉 0부터 -1까지는 "처음부터 끝까지 전부"를 의미합니다.
raw = run_redis("lrange", "recent:u1001", "0", "-1")
# repr()은 문자열을 "파이썬 코드로 쳐서 만들 수 있는 형태" 그대로 보여줍니다 — 줄바꿈이 \n으로 눈에 보이게 표시되어,
# raw 안에 실제로 어떤 문자(특히 줄바꿈)가 들어있는지 확인할 때 유용합니다.
print("원문(줄바꿈 문자열) ->", repr(raw))
# splitlines()는 문자열을 줄바꿈 기준으로 잘라 파이썬 리스트로 만들어줍니다 — repr()로 본 \n들이
# 실제로는 값과 값 사이의 구분자였다는 것을 이 결과로 확인할 수 있습니다.
print("splitlines() 적용 ->", raw.splitlines())


원문(줄바꿈 문자열) -> 'product:501\nproduct:502\nproduct:503'
splitlines() 적용 -> ['product:501', 'product:502', 'product:503']


In [32]:
# ✅ Step 1-B: Sorted Set — 문서(상품) 점수 랭킹 (4교시 예시를 실제로 실행)
run_redis("del", "doc:scores")
# zadd key 점수1 값1 점수2 값2 ...: 값마다 점수를 매겨 Sorted Set에 등록 — 자동으로 점수 순 정렬 상태가 유지됩니다.
run_redis("zadd", "doc:scores", "95.5", "A001", "87.2", "B002", "91.0", "C003")
# zrevrange: REV(역순) + range(범위) — 점수가 높은 것부터 내림차순으로, 0번째부터 -1번째(=끝)까지 전부 꺼냅니다.
# withscores를 붙이면 "값, 점수, 값, 점수, ..." 순서로 번갈아 반환됩니다.
raw = run_redis("zrevrange", "doc:scores", "0", "-1", "withscores")
print("원문(줄바꿈 문자열) ->")
print(raw)
# 💡 흔한 실수: 위에서 B002 점수가 87.200000000000003으로 보일 수 있습니다 — 부동소수점 표현 오차이며 실제로 자주 만나는 현상입니다.


원문(줄바꿈 문자열) ->
A001
95.5
C003
91
B002
87.2


## Step 2 · PostgreSQL 상품 테이블 + Cache-Aside 실측 (5교시 모듈 5-1)

In [33]:
# ✅ 실제 PostgreSQL에 상품 테이블 준비
# run_sql()에 세 개의 큰따옴표(""")로 감싼 여러 줄 문자열을 전달하면, 그 안의 SQL 문장 여러 개를
# 한 번에 실행할 수 있습니다(psql이 세미콜론(;)으로 문장을 하나씩 구분해서 처리합니다).
run_sql("""
DROP TABLE IF EXISTS products;  -- 같은 이름의 테이블이 이미 있으면 먼저 지우고 새로 시작(실습 재실행 대비)
CREATE TABLE products (
    id INT PRIMARY KEY,   -- 정수, 각 행을 구분하는 고유 값(중복 불가)
    name TEXT,            -- 문자열, 길이 제한 없음
    price INT,
    stock INT
);
INSERT INTO products VALUES
  (501, '무선 이어폰', 89000, 120),
  (502, '기계식 키보드', 145000, 45),
  (503, '휴대용 모니터', 320000, 12);
""", as_df=False)  # as_df=False: 결과를 표(DataFrame)로 받을 필요 없는 명령(CREATE/INSERT)이므로 텍스트로만 받음
run_sql("SELECT * FROM products;")  # 여긴 as_df 기본값(True) 그대로 사용 -> 결과가 pandas 표로 예쁘게 출력됨


,id,name,price,stock
0,501,무선 이어폰,89000,120
1,502,기계식 키보드,145000,45
2,503,휴대용 모니터,320000,12


In [34]:
# ✅ Cache-Aside 구현 + 실제 DB 호출 횟수 세기
# Cache-Aside(캐시 옆에 끼워두기) 패턴: "캐시에 있으면 캐시에서, 없으면 DB에서 가져와 캐시에 저장" 하는 방식입니다.
# DB_CALL_COUNT를 딕셔너리로 만든 이유: 아래 fetch_from_db() 함수 안에서 값을 바꿔도, 함수 밖 이 변수에
# 그 변경이 그대로 반영되게 하기 위해서입니다(파이썬 함수는 보통 바깥 변수를 직접 못 바꾸지만,
# 딕셔너리처럼 "안에 든 내용"을 바꾸는 건 가능합니다 — count라는 key의 값만 갱신하는 방식).
DB_CALL_COUNT = {"count": 0}

def fetch_from_db(product_id):
    """실제로 DB를 조회한 횟수를 세기 위한 래퍼."""
    DB_CALL_COUNT["count"] += 1  # 이 함수가 호출될 때마다(=진짜 DB에 쿼리를 날릴 때마다) 1씩 증가
    # f"...{product_id}..." 처럼 문자열 앞에 f를 붙이고 중괄호 {} 안에 변수를 넣으면, 그 변수의 값이
    # 문자열 안에 그대로 끼워집니다(f-string) — 예: product_id가 501이면 "WHERE id = 501"이 됩니다.
    df = run_sql(f"SELECT id, name, price, stock FROM products WHERE id = {product_id};")
    if df.empty:  # .empty는 pandas 표(DataFrame)에 행이 하나도 없을 때 True가 되는 속성
        return None
    # .iloc[0]은 표의 "0번째 행"을 선택하고, .to_dict()는 그 행을 {컬럼명: 값} 형태의 딕셔너리로 바꿔줍니다.
    return df.iloc[0].to_dict()

def get_product_cache_aside(product_id, ttl=30):
    key = f"product:{product_id}"
    cached = run_redis("get", key)  # 먼저 캐시(Redis)부터 확인
    if cached:  # cached가 None이 아니고 빈 문자열도 아니면(=캐시에 값이 있으면) True로 취급됨
        # json.loads()는 JSON 형식의 문자열을 파이썬 딕셔너리로 되돌리는 함수입니다
        # (아래에서 json.dumps()로 저장한 것을 다시 꺼낼 때 짝을 이룹니다).
        return "CACHE_HIT", json.loads(cached)

    row = fetch_from_db(product_id)  # 캐시에 없으면 실제 DB 조회(이때만 DB_CALL_COUNT가 늘어남)
    if row is None:
        return "DB_MISS", None

    # json.dumps(row, ensure_ascii=False)는 파이썬 딕셔너리를 JSON 문자열로 바꿉니다.
    # ensure_ascii=False를 주는 이유: 기본값(True)이면 한글이 \uXXXX 같은 이스케이프 코드로 바뀌어 버리는데,
    # False로 하면 한글을 그대로(읽을 수 있는 형태로) 저장합니다.
    # "EX", str(ttl)은 4교시에서 본 것과 같은 만료 시간(초) 설정 — ttl=30이면 30초 뒤 자동 삭제됩니다.
    run_redis("set", key, json.dumps(row, ensure_ascii=False), "EX", str(ttl))
    return "CACHE_MISS_DB_LOAD", row

run_redis("del", "product:501")  # 이전 실습의 캐시 흔적을 지우고 깨끗하게 시작
for i in range(4):  # 같은 상품(501)을 4번 연속 요청 — 캐시가 잘 동작하면 처음 1번만 DB를 타야 함
    status, data = get_product_cache_aside(501)
    # i+1인 이유: range(4)는 0,1,2,3을 만들어내므로(0부터 시작), 사람이 읽기 편하게 1번째~4번째로 표시
    print(f"{i+1}번째 요청 -> {status}")

print()
print("실제 DB 호출 횟수:", DB_CALL_COUNT["count"], "/ 요청 4번 중")
print("-> 처음 한 번만 DB를 타고, 나머지는 Redis 캐시로 응답했습니다.")


1번째 요청 -> CACHE_MISS_DB_LOAD
2번째 요청 -> CACHE_HIT
3번째 요청 -> CACHE_HIT
4번째 요청 -> CACHE_HIT

실제 DB 호출 횟수: 1 / 요청 4번 중
-> 처음 한 번만 DB를 타고, 나머지는 Redis 캐시로 응답했습니다.


## Step 3 · mongomock으로 임베딩 vs 참조 (5교시 모듈 5-2)

> ⚠️ **pymongo 실제 API와 동일**: `insert_one()`은 도큐먼트가 아니라 `InsertOneResult`를 반환합니다 — id는 `.inserted_id`로 꺼냅니다. 지난 버전의 `FakeMongoCollection`은 id를 바로 반환하는 손수 제작 방식이었지만, mongomock은 실제 pymongo와 똑같이 동작합니다.

In [35]:
# ✅ 임베딩(embedding) 설계: 관련 데이터를 문서 하나에 다 넣는 방식
users_embedded = mongo_db["users_embedded"]  # mongo_db["이름"]은 SQL의 "테이블"에 해당하는 컬렉션(collection)을 선택/생성
# insert_one({...})은 딕셔너리 하나를 문서(document) 하나로 저장합니다 — SQL의 한 행(row) 저장과 비슷합니다.
# .inserted_id는 insert_one()의 반환값 중 "방금 저장된 문서의 _id"만 꺼내는 것입니다 — MongoDB는 문서를 저장하면
# _id라는 고유 식별자를 자동으로 만들어주는데(SQL의 PRIMARY KEY와 비슷한 역할), 이후 이 uid로 그 문서를 다시 찾습니다.
uid = users_embedded.insert_one({
    "name": "박서연",
    "email": "seoyeon@example.com",
    "profile": {"dept": "AI캠퍼스", "level": "silver"},  # 값으로 딕셔너리를 또 넣을 수 있음 — "문서 안에 문서"(중첩 구조)
    "recent_views": ["product:501"],  # 값으로 리스트도 가능 — 이 배열에 나중에 항목을 계속 추가할 예정
}).inserted_id
# update_one(찾을 조건, 변경 내용): 첫 인자로 "_id가 uid인 문서"를 찾고, 두 번째 인자로 무엇을 바꿀지 지정합니다.
# "$push"는 MongoDB의 "업데이트 연산자" 중 하나로, 배열 필드(recent_views)의 맨 뒤에 값을 하나 추가합니다
# — 파이썬 리스트의 .append()와 같은 동작을, 이미 저장된 문서에 대해 수행하는 것입니다.
users_embedded.update_one({"_id": uid}, {"$push": {"recent_views": "product:502"}})
users_embedded.update_one({"_id": uid}, {"$push": {"recent_views": "product:503"}})
# "$set"은 필드 값을 통째로 덮어쓰는 연산자입니다. "profile.level"처럼 점(.)으로 이어 쓰면 중첩된 딕셔너리
# 안쪽의 필드(profile 안의 level)까지 정확히 지정해서 그 부분만 바꿀 수 있습니다(profile 전체를 안 건드림).
users_embedded.update_one({"_id": uid}, {"$set": {"profile.level": "gold"}})

print("임베딩 방식 (문서 하나로 다 조회됨):")
# find_one({조건})은 조건에 맞는 문서 딱 하나를 찾아 딕셔너리로 반환합니다(SQL의 SELECT ... LIMIT 1과 비슷).
# json.dumps(..., indent=2)는 딕셔너리를 사람이 읽기 좋은 들여쓰기 있는 JSON 문자열로 바꿔주고,
# default=str은 JSON으로 직접 표현할 수 없는 값(MongoDB의 _id 같은 특수 타입)을 문자열로 변환해 오류를 막습니다.
print(json.dumps(users_embedded.find_one({"_id": uid}), ensure_ascii=False, indent=2, default=str))


임베딩 방식 (문서 하나로 다 조회됨):
{
  "name": "박서연",
  "email": "seoyeon@example.com",
  "profile": {
    "dept": "AI캠퍼스",
    "level": "gold"
  },
  "recent_views": [
    "product:501",
    "product:502",
    "product:503"
  ],
  "_id": "22b8c352-baeb-11f1-8478-e4c7676116dc"
}


In [36]:
# ✅ 참조(reference) 설계: 조회 기록을 별도 컬렉션으로 분리
# 방금 위 셀의 "임베딩"과 반대로, 이번엔 사용자 정보와 조회 기록을 서로 다른 두 컬렉션(테이블에 해당)에 나눠 담습니다
# — SQL로 치면 users 테이블과 view_history 테이블을 외래키(user_id)로 연결하는 것과 같은 발상입니다.
users_ref = mongo_db["users_ref"]
views_ref = mongo_db["views_ref"]

# 사용자 문서를 먼저 저장하고, 그 _id(uid2)를 "참조 키"로 삼아 조회 기록 문서들에 함께 저장합니다.
uid2 = users_ref.insert_one({"name": "박서연", "email": "seoyeon@example.com"}).inserted_id
# 위 임베딩 방식은 하나의 문서 안에 $push로 추가했지만, 여기서는 매번 별도의 새 문서(insert_one)로 저장합니다
# — "user_id": uid2가 SQL의 외래키(FOREIGN KEY)처럼 "이 기록이 누구 것인지"를 연결해줍니다.
views_ref.insert_one({"user_id": uid2, "product_id": "product:501"})
views_ref.insert_one({"user_id": uid2, "product_id": "product:502"})
views_ref.insert_one({"user_id": uid2, "product_id": "product:503"})

user_doc = users_ref.find_one({"_id": uid2})  # 1번째 조회: 사용자 정보만 가져옴
user_views = list(views_ref.find({"user_id": uid2}))   # 💡 find()는 커서를 반환 -> list()로 소진
# 위 주석의 "커서(cursor)"란: find()는 결과를 한 번에 다 메모리에 올리지 않고 "필요할 때마다 하나씩 꺼내는"
# 지연 평가 객체를 반환합니다(find_one은 하나만 찾으니 바로 값을 반환하지만, find는 여러 개일 수 있어 다릅니다).
# list(...)로 감싸면 그 커서를 끝까지 순회하면서 실제 값을 전부 꺼내 파이썬 리스트로 만듭니다("소진"시킨다고 표현).

print("참조 방식 (2번 조회해서 조합해야 함):")
print("  users_ref  :", user_doc)
print("  views_ref  :", user_views)
print()
print("-> 임베딩은 find_one 1번, 참조는 find_one + find로 2번입니다.")


참조 방식 (2번 조회해서 조합해야 함):
  users_ref  : {'name': '박서연', 'email': 'seoyeon@example.com', '_id': ObjectId(2fefb793-baeb-11f1-8f67-e4c7676116dc)}
  views_ref  : [{'user_id': ObjectId(2fefb793-baeb-11f1-8f67-e4c7676116dc), 'product_id': 'product:501', '_id': ObjectId(2fefbab4-baeb-11f1-85ce-e4c7676116dc)}, {'user_id': ObjectId(2fefb793-baeb-11f1-8f67-e4c7676116dc), 'product_id': 'product:502', '_id': ObjectId(2fefbc54-baeb-11f1-a344-e4c7676116dc)}, {'user_id': ObjectId(2fefb793-baeb-11f1-8f67-e4c7676116dc), 'product_id': 'product:503', '_id': ObjectId(2fefbd84-baeb-11f1-9b2b-e4c7676116dc)}]

-> 임베딩은 find_one 1번, 참조는 find_one + find로 2번입니다.


## Step 4 · 6교시: 같은 질문, 세 DB에서 던지기

In [37]:
# ✅ 체크포인트 1 · PostgreSQL (실제 실행)
run_sql("""
DROP TABLE IF EXISTS view_history;
CREATE TABLE view_history (
    id SERIAL PRIMARY KEY,   -- SERIAL: INSERT할 때마다 1,2,3...으로 자동 증가하는 정수(직접 값을 안 넣어도 됨)
    user_id INT,
    product_id TEXT,
    viewed_at TIMESTAMP DEFAULT now()   -- DEFAULT now(): INSERT 시 이 컬럼 값을 안 주면 "현재 시각"이 자동으로 채워짐
);
INSERT INTO view_history (user_id, product_id) VALUES
  (1001, 'product:501'), (1001, 'product:502'), (1001, 'product:503');
""", as_df=False)
# 위 INSERT는 id와 viewed_at 컬럼을 지정하지 않았지만, id는 SERIAL이라 자동 채번되고
# viewed_at은 DEFAULT now()라 자동으로 현재 시각이 들어갑니다 — 두 컬럼 다 생략 가능했던 이유입니다.

print("[관계형/PostgreSQL] - 실제 실행")
print(run_sql("""
SELECT user_id, product_id, viewed_at
FROM view_history
WHERE user_id = 1001
ORDER BY viewed_at DESC;
"""))


[관계형/PostgreSQL] - 실제 실행
   user_id   product_id                   viewed_at
0     1001  product:501  2026-09-28 04:42:21.285512
1     1001  product:502  2026-09-28 04:42:21.285512
2     1001  product:503  2026-09-28 04:42:21.285512


In [38]:
# ✅ 체크포인트 2 · MongoDB(mongomock) — Step 3에서 만든 users_embedded 재사용
# 위쪽 23번 셀에서 만든 users_embedded 컬렉션과 uid 변수를 그대로 다시 사용합니다(새로 만들지 않음).
print(json.dumps(users_embedded.find_one({"_id": uid}), ensure_ascii=False, indent=2, default=str))


{
  "name": "박서연",
  "email": "seoyeon@example.com",
  "profile": {
    "dept": "AI캠퍼스",
    "level": "gold"
  },
  "recent_views": [
    "product:501",
    "product:502",
    "product:503"
  ],
  "_id": "22b8c352-baeb-11f1-8478-e4c7676116dc"
}


In [39]:
# ✅ 체크포인트 3 · Redis (실제 실행 — Step 1에서 이미 생성)
# 앞의 Step 1-A(17번 셀)에서 만들어 둔 "recent:u1001" List를 새로 만들지 않고 그대로 다시 조회합니다.
print("[키값형/Redis] - 실제 실행, List 자료구조 그대로 재사용")
print(run_redis("lrange", "recent:u1001", "0", "-1").splitlines())


[키값형/Redis] - 실제 실행, List 자료구조 그대로 재사용
['product:501', 'product:502', 'product:503']


In [40]:
# ✅ 체크포인트 4 · 3 DB 비교
# 같은 "사용자 1001의 최근 조회 상품" 데이터를 관계형(PostgreSQL)·문서형(MongoDB)·키값형(Redis) 세 가지
# 방식으로 각각 어떻게 저장·조회했는지 한 화면에 모아, 세 모델의 결과와 특징을 직접 비교해봅니다.
print("[관계형/PostgreSQL] - 실제 실행")
print(run_sql("""
SELECT user_id, product_id, viewed_at
FROM view_history WHERE user_id = 1001 ORDER BY viewed_at DESC;
"""))
print()

print("[문서형/MongoDB(mongomock)] - 문서 하나에 배열로 보관")
print(json.dumps(users_embedded.find_one({"_id": uid}), ensure_ascii=False, indent=2, default=str))
print()

print("[키값형/Redis] - 실제 실행, List 자료구조 그대로 재사용")
print(run_redis("lrange", "recent:u1001", "0", "-1").splitlines())

print()
print("정리:")
print("  관계형   -> 스키마 고정, JOIN/집계 강함, '언제 조회했는지' 등 조건 검색이 쉬움")
print("  문서형   -> 스키마 유연, 사용자 하나 읽을 때 통째로 빠름, 조건 검색은 상대적으로 약함")
print("  키값형   -> 최신 N개 같은 단순 순서 접근이 압도적으로 빠름, 복잡한 조회는 못 함(용도 특화)")


[관계형/PostgreSQL] - 실제 실행
   user_id   product_id                   viewed_at
0     1001  product:501  2026-09-28 04:42:21.285512
1     1001  product:502  2026-09-28 04:42:21.285512
2     1001  product:503  2026-09-28 04:42:21.285512

[문서형/MongoDB(mongomock)] - 문서 하나에 배열로 보관
{
  "name": "박서연",
  "email": "seoyeon@example.com",
  "profile": {
    "dept": "AI캠퍼스",
    "level": "gold"
  },
  "recent_views": [
    "product:501",
    "product:502",
    "product:503"
  ],
  "_id": "22b8c352-baeb-11f1-8478-e4c7676116dc"
}

[키값형/Redis] - 실제 실행, List 자료구조 그대로 재사용
['product:501', 'product:502', 'product:503']

정리:
  관계형   -> 스키마 고정, JOIN/집계 강함, '언제 조회했는지' 등 조건 검색이 쉬움
  문서형   -> 스키마 유연, 사용자 하나 읽을 때 통째로 빠름, 조건 검색은 상대적으로 약함
  키값형   -> 최신 N개 같은 단순 순서 접근이 압도적으로 빠름, 복잡한 조회는 못 함(용도 특화)


## 🔰 보너스 · 랭킹에 내 상품 추가해보기

In [41]:
# TODO: 원하는 상품 ID와 점수(조회수)로 바꿔보세요
MY_PRODUCT = "product:999"
MY_SCORE = 200

run_redis("del", "ranking:daily")
# zadd key 점수 값: 값 하나짜리 등록도 가능합니다(4교시/17번 셀처럼 여러 개를 한 번에 등록하는 것도 되고,
# 이렇게 zadd를 여러 번 나눠 호출해서 값을 하나씩 추가하는 것도 됩니다 — 둘 다 같은 Sorted Set에 쌓입니다).
run_redis("zadd", "ranking:daily", str(MY_SCORE), MY_PRODUCT)

# zrevrank key value: 이 값이 "점수 내림차순으로 몇 번째(0부터 시작)인지" 순위를 숫자로 반환합니다.
# 0부터 시작하므로(0등, 1등, 2등...) 사람이 읽는 "1등, 2등..."으로 바꾸려면 +1을 해줘야 합니다.
rank = run_redis("zrevrank", "ranking:daily", MY_PRODUCT)
print(f"{MY_PRODUCT} 현재 순위: {int(rank) + 1}등")  # rank는 문자열로 오므로 int()로 숫자로 바꾼 뒤 +1
print("전체 순위:", run_redis("zrevrange", "ranking:daily", "0", "-1", "withscores"))


product:999 현재 순위: 1등
전체 순위: product:999
200
